# Fake Product Review Identification
This notebook loads the fake reviews dataset, preprocesses the text, trains a Logistic Regression model (Algorithm 1), and displays a Graphical User Interface (GUI) to test new reviews.

In [3]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import tkinter as tk
from tkinter import messagebox

# Download stopwords if not already downloaded
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
    
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [4]:
# Load the dataset
# Ensure the CSV is in the same directory as this notebook
df = pd.read_csv('fake reviews dataset.csv')

# The labels are 'CG' (Computer Generated/Fake) and 'OR' (Original/Real)
print("Dataset Shape:", df.shape)
print("Label Distribution:\n", df['label'].value_counts())
df.head()

Dataset Shape: (40432, 4)
Label Distribution:
 label
CG    20216
OR    20216
Name: count, dtype: int64


,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


In [5]:
# Preprocessing function
def preprocess_text(text):
    text = text.lower() # Lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove punctuation
    words = text.split()
    words = [w for w in words if w not in stop_words] # Remove stopwords
    return ' '.join(words)

# Apply preprocessing
print("Preprocessing text...")
df['clean_text'] = df['text_'].apply(preprocess_text)
print("Preprocessing done.")
df[['text_', 'clean_text']].head()

Preprocessing text...
Preprocessing done.


,text_,clean_text
0,"Love this! Well made, sturdy, and very comfor...",love well made sturdy comfortable love itvery ...
1,"love it, a great upgrade from the original. I...",love great upgrade original ive mine couple years
2,This pillow saved my back. I love the look and...,pillow saved back love look feel pillow
3,"Missing information on how to use it, but it i...",missing information use great product price
4,Very nice set. Good quality. We have had the s...,nice set good quality set two months


In [6]:
# Feature Extraction and Train-Test Split
X = df['clean_text']
y = df['label'] # CG = Fake, OR = Real

# TF-IDF Vectorization
# We limit to 5000 features to speed up training while maintaining good accuracy
vectorizer = TfidfVectorizer(max_features=5000)
X_vec = vectorizer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42)
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 32345
Test set size: 8087


In [7]:
# Model Training (Algorithm 1: Logistic Regression)
print("Training Logistic Regression Model...")
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("Training Complete.")

# Evaluation
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Training Logistic Regression Model...
Training Complete.
Accuracy: 0.880178063558798
Classification Report:
               precision    recall  f1-score   support

          CG       0.89      0.87      0.88      4016
          OR       0.87      0.89      0.88      4071

    accuracy                           0.88      8087
   macro avg       0.88      0.88      0.88      8087
weighted avg       0.88      0.88      0.88      8087

